# Notebook for training a time-series model to detect the moment of germination

## Importing libraries

In [ ]:
from pathlib import Path

import keras_tuner as kt
import numpy as np
import tensorflow as tf
from keras import Model
from keras.applications.efficientnet_v2 import EfficientNetV2B0, preprocess_input
from keras.callbacks import EarlyStopping, ReduceLROnPlateau, TensorBoard
from keras.layers import (
    LSTM,
    BatchNormalization,
    Dense,
    Flatten,
    Input,
    Lambda,
    TimeDistributed,
)
from keras.models import load_model
from matplotlib import pyplot as plt
from tcn import TCN
from tensorflow import keras  # type: ignore

# Advanced users: edit these paths directly in this notebook.
ROOT_PATH = Path("/home/vasakjakub/fenotypizace")
DATA_ROOT = ROOT_PATH / "data"
PREDICTION_MODEL_DIR = ROOT_PATH / "models" / "prediction_model"
NUMPY_DATASET_DIR = DATA_ROOT / "numpy_dataset"
LOG_ROOT = ROOT_PATH / "logs"

print("Repo root:", ROOT_PATH)
print("Prediction model dir:", PREDICTION_MODEL_DIR)
print("NumPy dataset dir:", NUMPY_DATASET_DIR)
print("TensorFlow GPUs:", tf.config.list_physical_devices("GPU"))

In [ ]:
print(tf.__version__)

## Setting up the paths to the folders

In [ ]:
# Repository-local paths
main_dir = str(ROOT_PATH) + "/"

data_path = f"{DATA_ROOT}/"
model_path = f"{PREDICTION_MODEL_DIR}/"
data_images_path = NUMPY_DATASET_DIR

# Image parameters
height, width, colors = 75, 75, 3

# Step size used to localize the moment of germination precisely
slide_step = 1

# Default model hyperparameters
num_epochs = 300

## Printing function to visualize the training curves

In [ ]:
def visualize_training(model_history: keras.callbacks.History) -> None:
    plt.figure(figsize=(30, 10))
    plt.plot(np.array(model_history.history["loss"]), "r--", label="Train loss")
    plt.plot(np.array(model_history.history["accuracy"]), "g--", label="Train accuracy")
    plt.plot(np.array(model_history.history["val_loss"]), "r-", label="Validation loss")
    plt.plot(
        np.array(model_history.history["val_accuracy"]),
        "g-",
        label="Validation accuracy",
    )
    plt.title("Training session's progress over iterations")
    plt.legend(loc="upper right")
    plt.ylabel("Training Progress (Loss/Accuracy)")
    plt.xlabel("Training Epoch")
    plt.ylim(0)
    plt.show()

## Saving/loading functions

In [ ]:
def save_model(dir_name: str, model_name: Model) -> None:
    directory = PREDICTION_MODEL_DIR / dir_name
    directory.mkdir(parents=True, exist_ok=True)

    model_name.save(directory)
    print(f"Model: {dir_name} was saved.")

In [ ]:
def load_data_npy(
    base_path: Path,
    time_step: int,
    size: int,
    sample: str,
) -> tuple[np.ndarray, np.ndarray]:
    train_sample = np.load(base_path / f"data_{height}_{time_step}_{size}_{sample}.npy")
    annotation_sample = np.load(
        base_path / f"data_{height}_{time_step}_{size}_{sample}_ann.npy"
    )

    unique_values, counts = np.unique(annotation_sample, return_counts=True)
    print(
        f"Data has shape: {train_sample.shape} with annotations: {annotation_sample.shape}"
    )
    print(f"Unique annotation values: {unique_values} with counts: {counts}\n")

    return train_sample, annotation_sample

In [ ]:
def evaluate_model(
    time_step: int,
    size: int,
    model_name: Model | str,
    path: Path | None = None,
) -> None:
    x_test, y_test = load_data_npy(
        base_path=data_images_path, time_step=time_step, sample="test", size=size
    )

    if path is None:
        loaded_model = model_name
    else:
        if not isinstance(model_name, str):
            raise TypeError("model_name must be a string when path is provided.")
        loaded_model = load_model(str(path / model_name))

    evaluation = loaded_model.evaluate(x_test, y_test)  # type: ignore[attr-defined]

    print("Evaluation Loss:", evaluation[0])
    print("Evaluation Accuracy:", evaluation[1])

In [ ]:
def create_callbacks(mode: str) -> EarlyStopping:
    if mode == "tuner":
        return EarlyStopping(
            monitor="val_loss", mode="min", patience=3, restore_best_weights=True
        )
    return EarlyStopping(
        monitor="val_loss", mode="min", patience=5, restore_best_weights=True
    )

## Model functions

In [ ]:
def tcn_layer_tuner(hp: kt.HyperParameters) -> TCN:
    nb_filters = hp.Int("nb_filters", min_value=4, max_value=30, step=2)
    kernel_size = hp.Choice("kernel_size", values=[2, 3, 4, 5, 6])
    dropout_rate = hp.Float("dropout_rate", min_value=0.0, max_value=0.05, step=0.01)
    nb_stacks = hp.Int("nb_stacks", min_value=1, max_value=2, step=1)

    return TCN(
        nb_filters=nb_filters,
        kernel_size=kernel_size,
        nb_stacks=nb_stacks,
        dilations=(1, 2, 4),
        padding="causal",
        use_skip_connections=True,
        dropout_rate=dropout_rate,
        return_sequences=False,
        activation="sigmoid",
        kernel_initializer="he_normal",
        use_batch_norm=False,
        use_layer_norm=False,
        use_weight_norm=False,
    )

In [ ]:
def lstm_layer_tuner(hp: kt.HyperParameters) -> LSTM:
    units = hp.Choice("lstm_units", values=[32, 64, 128, 256, 512])
    return LSTM(units, return_sequences=False)

In [ ]:
def define_model_efficientnet_b0(
    hp: kt.HyperParameters,
    time_steps: int,
    h: int,
    w: int,
    c: int,
    tuner_type: str,
) -> Model:
    if tuner_type == "tcn_tuner":
        time_layer = tcn_layer_tuner(hp)
    elif tuner_type == "lstm_tuner":
        time_layer = lstm_layer_tuner(hp)
    else:
        raise ValueError("Enter a correct function")

    shape = (time_steps, h, w, c)
    inputs = Input(shape=shape)

    x = TimeDistributed(Lambda(lambda img: preprocess_input(img)))(inputs)

    base_cnn = EfficientNetV2B0(
        input_shape=(h, w, c), include_top=False, weights="imagenet"
    )
    x = TimeDistributed(base_cnn)(x)
    x = TimeDistributed(BatchNormalization())(x)
    x = TimeDistributed(Flatten())(x)
    x = time_layer(x)
    x = Dense(1, activation="sigmoid")(x)

    return Model(inputs=[inputs], outputs=[x])

In [ ]:
def build_model(
    hp: kt.HyperParameters,
    tuner_type: str,
    time_step: int,
    height: int,
    width: int,
    colors: int,
) -> Model:
    model = define_model_efficientnet_b0(
        hp, time_step, height, width, colors, tuner_type
    )
    learning_rate = hp.Choice(
        "learning_rate",
        values=[
            0.0005,
            0.0001,
            0.00001,
            0.00005,
        ],
    )
    optimizer = keras.optimizers.Adam(learning_rate=learning_rate)
    model.compile(optimizer, loss="binary_crossentropy", metrics=["accuracy"])

    return model

In [ ]:
def create_tuner(
    tuner_type: str,
    time_step: int,
    size: int,
    height: int,
    width: int,
    colors: int,
) -> kt.Hyperband | None:
    if tuner_type not in {"tcn_tuner", "lstm_tuner"}:
        print("Choose a valid temporal layer type.")
        return None

    project = f"keras_{tuner_type}_{time_step}_{size}"

    return kt.Hyperband(
        lambda hp: build_model(hp, tuner_type, time_step, height, width, colors),
        objective="val_accuracy",
        max_epochs=10,
        factor=3,
        directory=str(PREDICTION_MODEL_DIR / "keras_tuner_dir"),
        project_name=project,
        overwrite=True,
    )

# Main code

## Setting the time window and loading data

In [ ]:
time_step = 2
size = 8

data_train, annotations_train = load_data_npy(
    base_path=data_images_path, time_step=time_step, sample="train", size=size
)

data_val, annotations_val = load_data_npy(
    base_path=data_images_path, time_step=time_step, sample="val", size=size
)

## TCN model training

In [ ]:
tuner_tcn = create_tuner("tcn_tuner", time_step, size, height, width, colors)

In [ ]:
tuner_tcn.search(  # type: ignore
    data_train,
    annotations_train,
    epochs=num_epochs,
    validation_data=(data_val, annotations_val),
    callbacks=[
        create_callbacks("tuner"),
    ],
    use_multiprocessing=True,
    workers=8,
)

In [ ]:
best_hps_tcn = tuner_tcn.get_best_hyperparameters(num_trials=1)[0]  # type: ignore

best_hyperparameters_tcn = tuner_tcn.get_best_hyperparameters()[0]  # type: ignore
print("Best hyperparameters:", best_hyperparameters_tcn.values)

In [ ]:
callbacks = [
    create_callbacks("train"),
    ReduceLROnPlateau(monitor="val_loss", factor=0.2, patience=3, min_lr=1e-6),  # type: ignore
    TensorBoard(log_dir=str(LOG_ROOT / "training" / f"tcn_{time_step}_{size}")),
]

model_tcn = tuner_tcn.hypermodel.build(best_hps_tcn)  # type: ignore
model_tcn.summary()
history_tcn = model_tcn.fit(
    data_train,
    annotations_train,
    epochs=num_epochs,
    validation_data=(data_val, annotations_val),
    callbacks=callbacks,
)

In [ ]:
visualize_training(history_tcn)

In [ ]:
save_model(f"ef2_tcn_tuner_{time_step}-{size}", model_tcn)

In [ ]:
evaluate_model(time_step, size, model_tcn)

## LSTM model training

In [ ]:
tuner_lstm = create_tuner("lstm_tuner", time_step, size, height, width, colors)

In [ ]:
tuner_lstm.search(  # type: ignore
    data_train,
    annotations_train,
    epochs=10,
    validation_data=(data_val, annotations_val),
    callbacks=[create_callbacks("tuner")],
)

In [ ]:
best_hps_lstm = tuner_lstm.get_best_hyperparameters(num_trials=1)[0]  # type: ignore

best_hyperparameters_lstm = tuner_lstm.get_best_hyperparameters()[0]  # type: ignore
print("Best hyperparameters:", best_hyperparameters_lstm.values)

In [ ]:
callbacks = [
    create_callbacks("train"),
    ReduceLROnPlateau(monitor="val_loss", factor=0.2, patience=3, min_lr=1e-6),  # type: ignore
    TensorBoard(log_dir=str(LOG_ROOT / "training" / f"lstm_{time_step}_{size}")),
]

model_lstm = tuner_lstm.hypermodel.build(best_hps_lstm)  # type: ignore
model_lstm.summary()
history_lstm = model_lstm.fit(
    data_train,
    annotations_train,
    epochs=num_epochs,
    validation_data=(data_val, annotations_val),
    callbacks=callbacks,
)

In [ ]:
visualize_training(history_lstm)

In [ ]:
save_model(f"ef2_lstm_tuner_{time_step}-{size}_all_{height}", model_lstm)

In [ ]:
evaluate_model(time_step, size, model_lstm)

# Fine-tuning the pretrained model

In [ ]:
import datetime

from keras.callbacks import ModelCheckpoint

In [ ]:
# Load the pretrained model
model = load_model(str(PREDICTION_MODEL_DIR / "ef2_tcn_2-8_org_nab_train_relearned"))

In [ ]:
# Define callbacks for fine-tuning
fine_tune_checkpoint_dir = PREDICTION_MODEL_DIR / "ef2_tcn_2-8_org_nab_train_finetuned"

callbacks = [
    EarlyStopping(
        monitor="val_loss", mode="min", patience=5, restore_best_weights=True, verbose=1
    ),
    ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.2,
        patience=3,
        min_lr=1e-6,
        verbose=1,  # type: ignore
    ),
    ModelCheckpoint(
        filepath=str(
            fine_tune_checkpoint_dir / "checkpoint-{epoch:02d}-{val_loss:.4f}"
        ),
        monitor="val_loss",
        save_best_only=True,
        mode="min",
        verbose=1,
    ),
    TensorBoard(
        log_dir=str(
            LOG_ROOT
            / "fine_tuning"
            / datetime.datetime.now(tz=datetime.UTC).strftime("%Y%m%d-%H%M%S")
        )
    ),
]

model.compile(  # type: ignore
    optimizer=keras.optimizers.Adam(learning_rate=1e-5),
    loss="binary_crossentropy",
    metrics=["accuracy"],
)

history = model.fit(  # type: ignore
    data_train,
    annotations_train,
    validation_data=(data_val, annotations_val),
    epochs=50,
    batch_size=32,
    callbacks=callbacks,
    verbose=1,
)